# 19.2 Capstone — Log Analysis Pipeline

**Prerequisites:** 03 Flow Control (3.3 comprehensions), 04 Functions (4.3 generators),
07 Modules (7.1 pathlib, 7.3 itertools/functools), 08 File Handling (8.2 CSV, 8.3 JSON),
09 Regular Expression (9.1), 15 Testing and Debugging (15.3 pytest, 15.10 logging)
**Target:** Python 3.12+

### What you'll build

A complete, streaming **log-analysis pipeline** — the kind of tool every ops and backend
team ends up writing:

```
access.log ──► parse (regex) ──► filter / normalise ──► aggregate ──► report.csv
                    │                (generators)        (Counter,     report.jsonl
                    └─► malformed → logging.warning       quantiles)
```

- Parse a web-server access log with a **compiled, named-group regex** (**9.1**)
- Stream it through a **generator pipeline** — constant memory, early stopping (**4.3**)
- Survive dirty data: malformed lines are **logged and counted, never fatal** (**15.10**)
- Aggregate with `Counter`, `defaultdict` and `statistics.quantiles` (**2.5**, **7.3**)
- Emit **CSV** and **JSON Lines** artifacts (**8.2**, **8.3**)
- Prove the design with `tracemalloc` measurements and a **pytest** suite (**15.3**)

Everything runs in temp directories and is fully deterministic — no network, no `input()`,
and nothing is written into this repo. A cleanup cell at the end removes every file.

## 1. The problem

It is 09:40 and the on-call channel asks the eternal question:

> **"Which endpoints are erroring, and how slow are they?"**

What you have is an **access log** — one line per HTTP request, written by the web server:

```
10.0.3.7 - - [01/Aug/2026:09:00:04 +0000] "GET /api/jobs/4821 HTTP/1.1" 200 1204 0.031
10.0.8.19 - alice [01/Aug/2026:09:00:11 +0000] "POST /api/jobs HTTP/1.1" 201 88 0.114
10.0.1.3 - - [01/Aug/2026:09:00:12 +0000] "GET /api/orders/55310/items HTTP/1.1" 503 44 12.508
```

Field by field: client IP, identity (`-`), user, `[timestamp]`, `"METHOD path HTTP/version"`,
status code, response bytes, and the request latency in **seconds**.

The answer the pipeline must produce:

1. an **error report** — how many 5xx per endpoint, and when they happened
2. a **latency summary** — p50 and p95 per endpoint
3. machine-readable artifacts (**CSV** for spreadsheets, **JSON Lines** for downstream tools)

Real logs are big (gigabytes) and dirty (truncated writes, garbage bytes, clients sending
junk). Both facts drive the design decisions in section 3.

## 2. Generate the dataset — deterministically

A capstone that downloads a log breaks the moment the URL dies. Instead we **generate** a
realistic log with a **seeded** `random.Random` (an instance, not the module-level
functions, so nothing else can disturb the sequence): every run of this notebook produces
byte-for-byte identical data, so every number below is reproducible.

The generator deliberately builds in the problems the pipeline must handle:

- a **5xx burst** on one endpoint during the 11:00 hour (the incident we will find)
- paths carrying numeric IDs (`/api/jobs/4821`) that must be collapsed for aggregation
- search queries containing a **raw space** (`q=red shoes`) — clients really send these
- a handful of **malformed lines** (truncated writes, garbage) sprinkled through the file

Everything lands in a `tempfile.mkdtemp` directory (**7.1**) — nothing touches the repo.

In [ ]:
import csv
import json
import logging
import random
import re
import shutil
import statistics
import subprocess
import sys
import tempfile
import textwrap
import tracemalloc
from collections import Counter, defaultdict
from collections.abc import Iterable, Iterator
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from itertools import groupby, islice
from pathlib import Path

WORK = Path(tempfile.mkdtemp(prefix="py192_"))   # every artifact lives under here
LOG_PATH = WORK / "access.log"

# 15.10: configure logging once, at startup. stream=sys.stdout keeps warnings
# inline with the notebook output instead of in a red stderr block.
logging.basicConfig(level=logging.INFO, stream=sys.stdout, force=True,
                    format="%(levelname)s %(name)s: %(message)s")
pipe_log = logging.getLogger("pipeline")

print("work dir:", WORK)
print("python  :", sys.version.split()[0])

In [ ]:
rng = random.Random(1729)                        # seeded -> identical every run
START = datetime(2026, 8, 1, 9, 0, 0, tzinfo=timezone.utc)

QUERIES = ["boots", "red shoes", "socks", "rain jacket", "belt"]


def make_request(ts: datetime) -> tuple[str, str, int, float]:
    """Return (method, path, status, latency_seconds) for one log line."""
    kind = rng.choices(
        ["list_jobs", "get_job", "post_job", "get_user", "order_items",
         "search", "login", "health"],
        weights=[22, 25, 8, 14, 12, 9, 6, 4])[0]

    method, path, base = {
        "list_jobs":   ("GET",  "/api/jobs",                                0.045),
        "get_job":     ("GET",  f"/api/jobs/{rng.randint(1000, 9999)}",     0.030),
        "post_job":    ("POST", "/api/jobs",                                0.120),
        "get_user":    ("GET",  f"/api/users/{rng.randint(100, 999)}",      0.035),
        "order_items": ("GET",  f"/api/orders/{rng.randint(10000, 99999)}/items", 0.180),
        "search":      ("GET",  f"/search?q={rng.choice(QUERIES)}",         0.250),
        "login":       ("POST", "/login",                                   0.090),
        "health":      ("GET",  "/health",                                  0.002),
    }[kind]

    status = 201 if method == "POST" and kind == "post_job" else 200
    if kind == "order_items" and ts.hour == 11 and rng.random() < 0.45:
        status = rng.choice([500, 502, 503])     # the 11:00 incident
    elif kind == "order_items" and rng.random() < 0.02:
        status = 500                             # background failure rate
    elif kind == "login" and rng.random() < 0.25:
        status = 401
    elif kind == "get_job" and rng.random() < 0.03:
        status = 404
    elif kind in {"list_jobs", "get_user", "search"} and rng.random() < 0.01:
        status = 500

    latency = (rng.uniform(2.5, 30.0) if status >= 500        # timeouts are SLOW
               else base * rng.lognormvariate(0.0, 0.4))
    return method, path, status, latency


ts = START
lines: list[str] = []
for _ in range(2_000):
    ts += timedelta(seconds=rng.expovariate(1 / 10.8))        # ~6h of traffic
    method, path, status, latency = make_request(ts)
    ip = f"10.0.{rng.randint(0, 9)}.{rng.randint(1, 254)}"
    user = rng.choice(["alice", "bob", "carol"]) if rng.random() < 0.1 else "-"
    size = 0 if status >= 500 and rng.random() < 0.3 else rng.randint(40, 4_000)
    lines.append(f'{ip} - {user} [{ts:%d/%b/%Y:%H:%M:%S +0000}] '
                 f'"{method} {path} HTTP/1.1" {status} {size} {latency:.3f}')

# Dirty data: truncated writes and garbage, inserted at deterministic positions.
malformed = ([line[:40] for line in rng.sample(lines, 6)]          # truncated
             + ["#$%! not a log line !%$#"] * 3                    # garbage
             + ["10.0.0.1 - - oops no timestamp",                  # partial writes
                '"POST /login HTTP/1.1" 401', "- - -"])
for pos in sorted(rng.sample(range(len(lines)), len(malformed)), reverse=True):
    lines.insert(pos, malformed.pop())

LOG_PATH.write_text("\n".join(lines) + "\n", encoding="utf-8")
TOTAL_LINES = len(lines)

print(f"wrote {TOTAL_LINES} lines ({LOG_PATH.stat().st_size / 1024:.0f} KiB), "
      f"12 of them deliberately malformed")
print(f"time span: {START:%H:%M} -> {ts:%H:%M} UTC\n")
print("\n".join(lines[:4]))

## 3. Design decisions — and why

Four choices shape everything that follows. Each earns its place with a demonstration,
not an assertion.

### 3.1 Why a regex with named groups, not `split()`

`line.split()` *looks* sufficient — the fields are space-separated, aren't they? They are
not: the quoted request `"GET /search?q=red shoes HTTP/1.1"` contains a space **inside** a
field, so every positional index after it silently shifts. Positional parsing doesn't
fail on such a line — worse, it **succeeds with wrong values**.

In [ ]:
raw = LOG_PATH.read_text(encoding="utf-8").splitlines()
ok_line = next(line for line in raw if "/api/jobs/" in line)
sneaky = next(line for line in raw if "red shoes" in line)

print(ok_line)
print(sneaky)
print()
# Column 8 "should be" the status code...
print(f"ok_line.split()[8] = {ok_line.split()[8]!r}   <- the status, as hoped")
print(f"sneaky.split()[8]  = {sneaky.split()[8]!r}   <- silently wrong!")

A regex with **named groups** (**9.1**) fixes both problems at once:

- the quoted request is matched as a unit — `"(?P<method>[A-Z]+) (?P<path>.*?) HTTP/…"` —
  so an embedded space cannot shift anything;
- a line that does not match returns `None`, an **explicit, checkable failure** instead of
  silently wrong columns.

We **compile once at module level** (`re.compile`) because the pattern runs thousands of
times, and use `re.VERBOSE` so it can be commented. Names (`m["status"]`) beat positional
groups (`m.group(6)`) for the same reason keyword arguments beat a tuple: the code that
*uses* the match stays readable when the pattern evolves.

⚠️ **Anchor it.** `match()` anchors at the start; the trailing `$` anchors the end, so a
line with trailing garbage is rejected rather than half-parsed. And every quantifier here
is bounded or non-greedy — the backtracking discipline from **9.1** (ReDoS) applies to log
parsers more than anywhere, because attackers *write* your logs.

In [ ]:
LOG_RE = re.compile(r"""
    ^
    (?P<ip>\d{1,3}(?:\.\d{1,3}){3})\s+       # client IP
    -\s+(?P<user>\S+)\s+                      # identity is always '-'; user may not be
    \[(?P<ts>[^\]]+)\]\s+                     # [01/Aug/2026:09:00:04 +0000]
    "(?P<method>[A-Z]+)\s
     (?P<path>.*?)\s                          # non-greedy: path may contain spaces
     HTTP/(?P<http_version>[\d.]+)"\s+
    (?P<status>\d{3})\s+
    (?P<size>\d+)\s+
    (?P<latency>\d+\.\d+)
    $
""", re.VERBOSE)

# The walrus (1.4) binds and tests in one step - the idiomatic parse-and-check.
if (m := LOG_RE.match(sneaky)) is not None:
    for name, value in m.groupdict().items():
        print(f"{name:14} {value!r}")

### 3.2 Parsed lines become typed records

`m.groupdict()` gives strings. Downstream code wants a **timestamp**, an **int** status
and a **float** latency — so conversion happens exactly once, at the parse boundary, and
everything after works with a typed, immutable `dataclass`:

- `frozen=True` — a record is a fact; nothing downstream may edit it
- `slots=True` — thousands of instances, so per-instance `__dict__`s are pure waste
- latency is converted to **milliseconds** here, so no later stage ever wonders about units

`parse_line` returns `LogRecord | None` — the `None` is the *contract* for "this line is
not parseable", which section 3.3 turns into a logged-and-skipped event.

In [ ]:
@dataclass(frozen=True, slots=True)
class LogRecord:
    ts: datetime
    ip: str
    method: str
    path: str
    status: int
    size: int
    latency_ms: float


def parse_line(line: str) -> LogRecord | None:
    """Parse one access-log line; None means the line is malformed."""
    if (m := LOG_RE.match(line)) is None:
        return None
    return LogRecord(
        ts=datetime.strptime(m["ts"], "%d/%b/%Y:%H:%M:%S %z"),
        ip=m["ip"],
        method=m["method"],
        path=m["path"],
        status=int(m["status"]),
        size=int(m["size"]),
        latency_ms=float(m["latency"]) * 1000,
    )


print(parse_line(ok_line))
print(parse_line("#$%! not a log line !%$#"))

### 3.3 Why a generator pipeline — and why bad lines must not crash it

**Memory.** A 10 GiB log does not fit in RAM; a generator pipeline (**4.3**) holds **one
record at a time** no matter how big the file is. Section 6 measures this with
`tracemalloc`.

**Early stopping.** "Show me the first five errors" should read a few hundred lines, not
two thousand — `islice` (**7.3**) stops the *entire* pipeline the moment it has enough,
because generators are lazy end to end.

**Dirty data is normal data.** One truncated line two hours into a batch job must not
throw away the other 1,999,999 lines. The policy (**15.10**): a malformed line is
`logging.warning`-ed **with its line number** (so a human can go look) and **counted**
(so the report can say how much was skipped) — and the pipeline keeps going. Crashing is
for programming errors, not for other people's data.

⚠️ Each pass hands `parse_records` a **fresh `Counter`** — stats belong to a pass, not to
the module, or a second pass would silently double every number.

In [ ]:
def read_lines(path: Path) -> Iterator[str]:
    """Yield non-empty lines; the file is never read whole."""
    with path.open(encoding="utf-8") as fh:
        for line in fh:
            if stripped := line.strip():
                yield stripped


def parse_records(lines: Iterable[str], stats: Counter[str]) -> Iterator[LogRecord]:
    """Parse lines into LogRecords; malformed lines are logged, counted, skipped."""
    for lineno, line in enumerate(lines, start=1):
        if (record := parse_line(line)) is None:
            stats["malformed"] += 1
            pipe_log.warning("line %d unparseable: %r", lineno, line[:48])
            continue
        stats["parsed"] += 1
        yield record


# Lazy end to end: this reads only as much of the file as 3 records need.
for record in islice(parse_records(read_lines(LOG_PATH), Counter()), 3):
    print(f"{record.ts:%H:%M:%S}  {record.status}  "
          f"{record.latency_ms:8.1f} ms  {record.method} {record.path}")

In [ ]:
# One full pass: the warnings fire once per bad line, the Counter keeps score.
stats: Counter[str] = Counter()
for _ in parse_records(read_lines(LOG_PATH), stats):
    pass

print(f"\nparsed {stats['parsed']}, skipped {stats['malformed']} "
      f"of {TOTAL_LINES} lines")

# The bad lines are audited above; further passes need not repeat the noise.
# Level control (15.10) - the *counter* still counts, only the log goes quiet.
pipe_log.setLevel(logging.ERROR)

### 3.4 Why `Counter` and `defaultdict` for aggregation

Aggregating per key with a plain dict means the `if key not in d:` dance on every line.
The `collections` types (**2.5**, **7.3**) delete that boilerplate:

- `Counter` — "how many per key", with `.most_common()` for free
- `defaultdict(list)` — "collect values per key", for the latency distributions

But first, the keys must be *worth* grouping by. `/api/jobs/4821` and `/api/jobs/7130`
are the same endpoint; aggregated raw, every request is its own group and the report is
useless. A `re.sub` collapses numeric path segments to a `{id}` placeholder, and the
query string is dropped — `/search?q=boots` and `/search?q=belt` are one endpoint too.

In [ ]:
_ID_SEGMENT = re.compile(r"/\d+(?=/|$)")        # a path segment that is all digits


def normalise_path(path: str) -> str:
    """Collapse IDs and drop the query string: /api/jobs/4821 -> /api/jobs/{id}."""
    return _ID_SEGMENT.sub("/{id}", path.partition("?")[0])


def endpoint(record: LogRecord) -> str:
    return f"{record.method} {normalise_path(record.path)}"


for path in ["/api/jobs/4821", "/api/orders/55310/items",
             "/search?q=red shoes", "/health"]:
    print(f"{path:28} -> {normalise_path(path)}")

## 4. The pipeline, assembled

Each stage is a generator (or a generator expression, **3.3**) that consumes the previous
one. Nothing runs until something iterates the far end — and then the whole chain streams.
`records()` is the reusable head of the pipeline: every analysis below re-reads the file
through it, and each pass stays at constant memory.

In [ ]:
def records(stats: Counter[str] | None = None) -> Iterator[LogRecord]:
    """A fresh streaming pass over the whole log."""
    return parse_records(read_lines(LOG_PATH),
                         stats if stats is not None else Counter())


status_mix = Counter(f"{record.status // 100}xx" for record in records())
worst = Counter(record.status for record in records() if record.status >= 400)

print("status mix :", dict(sorted(status_mix.items())))
print("errors     :", worst.most_common())

### 4.1 Per-endpoint latency — p50 and p95

Percentiles need the **whole distribution** per endpoint, so this is the one stage that
must accumulate: a `defaultdict(list)` keyed by endpoint fills as the stream goes by,
then `statistics.quantiles(n=100)` cuts each list into percentiles — index 49 is the
median (p50), index 94 is p95.

⚠️ **Why p95 and not the mean?** One 30-second timeout among a hundred 30 ms requests
barely moves p50 but explodes the mean; p95 tells you what your slowest real users see.
The table below shows exactly that shape for the incident endpoint.

In [ ]:
latencies: defaultdict[str, list[float]] = defaultdict(list)
hits: Counter[str] = Counter()
errors_5xx: Counter[str] = Counter()

for record in records():
    key = endpoint(record)
    latencies[key].append(record.latency_ms)
    hits[key] += 1
    if record.status >= 500:
        errors_5xx[key] += 1


def percentile(values: list[float], pct: int) -> float:
    return statistics.quantiles(values, n=100)[pct - 1]


REPORT = [
    {"endpoint": key,
     "requests": hits[key],
     "errors_5xx": errors_5xx[key],
     "error_rate": round(errors_5xx[key] / hits[key], 4),
     "p50_ms": round(percentile(values, 50), 1),
     "p95_ms": round(percentile(values, 95), 1)}
    for key, values in latencies.items()
]
REPORT.sort(key=lambda row: row["p95_ms"], reverse=True)

print(f"{'endpoint':32}{'req':>6}{'5xx':>6}{'rate':>8}{'p50 ms':>10}{'p95 ms':>10}")
for row in REPORT:
    print(f"{row['endpoint']:32}{row['requests']:>6}{row['errors_5xx']:>6}"
          f"{row['error_rate']:>8.1%}{row['p50_ms']:>10.1f}{row['p95_ms']:>10.1f}")

### 4.2 The error timeline — `itertools.groupby`

*When* did the 5xx happen? `groupby` (**7.3**) groups **adjacent** items only — that is
the classic trap. Group this log by *endpoint* without sorting and you get fragments:

In [ ]:
# ⚠️ The groupby trap: unsorted keys -> fragmented groups (7.3).
keys = [endpoint(record) for record in records()]
fragments = sum(1 for _ in groupby(keys))
print(f"groupby on unsorted endpoint keys: {fragments} groups "
      f"for {len(set(keys))} actual endpoints")

In [ ]:
# But the log is already sorted by TIME - so grouping by hour needs no sort,
# no materialisation, and stays streaming end to end.
error_hours = (record.ts.replace(minute=0, second=0)
               for record in records() if record.status >= 500)

print("5xx per hour")
for hour, group in groupby(error_hours):
    count = sum(1 for _ in group)
    print(f"  {hour:%H:%M}  {'#' * count}  {count}")

The incident is now legible: a burst of 5xx concentrated in the **11:00 hour**, on
`GET /api/orders/{id}/items` — the endpoint whose p95 dwarfs its p50 in the table above.
That, plus the artifacts below, *is* the answer to the on-call question.

⚠️ If the timeline had needed grouping by anything other than file order, the rule from
**7.3** applies: `sorted(...)` first, *then* `groupby` — and sorting materialises, so it
costs the memory the rest of the pipeline avoided. Keep grouping keys aligned with file
order when you can.

## 5. Emit the artifacts

Two formats, two audiences:

- **CSV** (`csv.DictWriter`, **8.2**) — for spreadsheets and humans. `newline=""` in
  `open()` as always, and the writer takes its column order from `fieldnames`.
- **JSON Lines** (**8.3**) — one JSON object per line, for machines. Downstream tools
  (`jq`, log shippers, a warehouse loader) can stream it line by line, append to it
  without re-parsing, and a truncated last line loses one record, not the whole file —
  exactly the properties a *pipeline output* should have. A single JSON array has none
  of them.

In [ ]:
CSV_PATH = WORK / "endpoint_report.csv"
FIELDS = ["endpoint", "requests", "errors_5xx", "error_rate", "p50_ms", "p95_ms"]

with CSV_PATH.open("w", encoding="utf-8", newline="") as fh:   # 8.2: newline=""
    writer = csv.DictWriter(fh, fieldnames=FIELDS)
    writer.writeheader()
    writer.writerows(REPORT)

print(CSV_PATH.read_text(encoding="utf-8"))

In [ ]:
JSONL_PATH = WORK / "endpoint_report.jsonl"

with JSONL_PATH.open("w", encoding="utf-8") as fh:
    for row in REPORT:
        fh.write(json.dumps(row) + "\n")       # one object per line - that's JSONL

# Read it back the way a downstream tool would: line by line, lazily.
with JSONL_PATH.open(encoding="utf-8") as fh:
    for line in islice(fh, 3):
        print(json.loads(line))
print(f"... {JSONL_PATH.stat().st_size} bytes, one record per line")

## 6. The honest measurement

Section 3.3 claimed two advantages for the generator pipeline. Claims are cheap;
`tracemalloc` is not. Both functions below compute the **same status mix** and are
asserted equal — the only difference is that one materialises the file and all its
records as lists first, the other streams.

In [ ]:
def status_mix_list() -> Counter[int]:
    """Materialise everything, then count."""
    all_lines = LOG_PATH.read_text(encoding="utf-8").splitlines()
    all_records = [parse_line(line) for line in all_lines]
    return Counter(r.status // 100 for r in all_records if r is not None)


def status_mix_stream() -> Counter[int]:
    """One record in flight at a time."""
    return Counter(r.status // 100 for r in records())


tracemalloc.start()
via_list = status_mix_list()
_, peak_list = tracemalloc.get_traced_memory()
tracemalloc.stop()

tracemalloc.start()
via_stream = status_mix_stream()
_, peak_stream = tracemalloc.get_traced_memory()
tracemalloc.stop()

assert via_list == via_stream                    # same answer, different cost
print(f"same result      : {dict(sorted(via_stream.items()))}")
print(f"peak, list-based : {peak_list / 1024:8.1f} KiB")
print(f"peak, streaming  : {peak_stream / 1024:8.1f} KiB")

In [ ]:
# Early stopping: "show me the first 5 errors" - how much of the file is read?
lines_consumed: Counter[str] = Counter()


def counted(lines: Iterable[str], counter: Counter[str]) -> Iterator[str]:
    for line in lines:
        counter["lines"] += 1
        yield line


first_errors = islice(
    (r for r in parse_records(counted(read_lines(LOG_PATH), lines_consumed),
                              Counter())
     if r.status >= 500),
    5)
for record in first_errors:
    print(f"{record.ts:%H:%M:%S}  {record.status}  {endpoint(record)}")

print(f"\nlines read: {lines_consumed['lines']} of {TOTAL_LINES}")

### What the numbers actually show

- **Peak memory:** the list-based pass holds the full file text, the list of its lines
  and ~2,000 `LogRecord` objects at once; the streaming pass holds one line and one
  record plus fixed overheads (I/O buffers, the `Counter`). On this small file that is
  a several-fold gap, as printed above — the important part is how it *scales*. The
  list peak grows linearly with file size; the streaming peak is a flat floor. At
  10 GiB the list version is dead and the stream has not noticed.
- **Early stopping:** the first five 5xx cost roughly a **third** of the file, not all
  of it — `islice` closed the whole chain the moment it had five. Why not less? 5xx are
  rare before the 11:00 burst, so five of them genuinely take ~700 lines; ask for the
  first five *requests* instead and the pipeline stops after five lines.
- What the numbers do **not** show: streaming is not *faster* per line — both passes do
  the same regex work. The win is bounded memory and time-to-first-answer, not CPU.

## 7. Tests

The pipeline's parts are plain functions, so they test like plain functions. Following
the pattern from **15.3**, the pipeline's core is written into a temp project as a proper
module (in a real repo this notebook would *import* that module rather than define the
code twice) with a pytest suite covering the three behaviours that must never regress:

1. the parser, on a good line, on the space-in-query line, and on garbage
2. the path normaliser's collapsing rules
3. one aggregation — error counting per endpoint

In [ ]:
PROJECT = Path(tempfile.mkdtemp(prefix="logpipe_", dir=WORK))

(PROJECT / "logpipe.py").write_text(textwrap.dedent('''
    """Core of the 19.2 log pipeline, as an importable module."""
    import re
    from collections import Counter

    LOG_RE = re.compile(
        r'^(?P<ip>\\d{1,3}(?:\\.\\d{1,3}){3})\\s+-\\s+(?P<user>\\S+)\\s+'
        r'\\[(?P<ts>[^\\]]+)\\]\\s+'
        r'"(?P<method>[A-Z]+)\\s(?P<path>.*?)\\sHTTP/(?P<http_version>[\\d.]+)"\\s+'
        r'(?P<status>\\d{3})\\s+(?P<size>\\d+)\\s+(?P<latency>\\d+\\.\\d+)$')

    _ID_SEGMENT = re.compile(r"/\\d+(?=/|$)")


    def parse_fields(line: str) -> dict[str, str] | None:
        m = LOG_RE.match(line)
        return None if m is None else m.groupdict()


    def normalise_path(path: str) -> str:
        return _ID_SEGMENT.sub("/{id}", path.partition("?")[0])


    def count_errors(field_dicts) -> Counter[str]:
        """5xx per normalised endpoint, malformed entries (None) skipped."""
        return Counter(
            f"{d['method']} {normalise_path(d['path'])}"
            for d in field_dicts
            if d is not None and int(d["status"]) >= 500)
'''), encoding="utf-8")

(PROJECT / "test_logpipe.py").write_text(textwrap.dedent('''
    import pytest
    from logpipe import count_errors, normalise_path, parse_fields

    GOOD = '10.0.3.7 - - [01/Aug/2026:09:00:04 +0000] "GET /api/jobs/4821 HTTP/1.1" 200 1204 0.031'
    SNEAKY = '10.0.1.9 - - [01/Aug/2026:09:02:00 +0000] "GET /search?q=red shoes HTTP/1.1" 500 88 3.100'


    def test_good_line_parses():
        fields = parse_fields(GOOD)
        assert fields is not None
        assert (fields["method"], fields["status"]) == ("GET", "200")
        assert fields["path"] == "/api/jobs/4821"


    def test_space_in_query_does_not_shift_fields():
        fields = parse_fields(SNEAKY)
        assert fields is not None
        assert fields["path"] == "/search?q=red shoes"
        assert fields["status"] == "500"        # split() got this one wrong


    @pytest.mark.parametrize("line", [
        "", "#$%! not a log line !%$#",
        GOOD[:40],                              # truncated write
        GOOD + " trailing-garbage",             # the $ anchor rejects this
    ])
    def test_malformed_returns_none(line):
        assert parse_fields(line) is None


    @pytest.mark.parametrize("raw, expected", [
        ("/api/jobs/4821", "/api/jobs/{id}"),
        ("/api/orders/55310/items", "/api/orders/{id}/items"),
        ("/search?q=red shoes", "/search"),
        ("/health", "/health"),
        ("/v2/things/12/parts/9", "/v2/things/{id}/parts/{id}"),
    ])
    def test_normalise_path(raw, expected):
        assert normalise_path(raw) == expected


    def test_count_errors_skips_malformed_and_groups_ids():
        parsed = [parse_fields(GOOD), parse_fields(SNEAKY), None,
                  parse_fields(SNEAKY.replace("q=red shoes", "q=belt"))]
        assert count_errors(parsed) == {"GET /search": 2}
'''), encoding="utf-8")

result = subprocess.run(
    [sys.executable, "-m", "pytest", "--no-header", "-p", "no:cacheprovider", "-v"],
    cwd=PROJECT, capture_output=True, text=True,
    encoding="utf-8", errors="replace", timeout=300)
print(result.stdout.rstrip())
print("exit code:", result.returncode)

---

## Common Mistakes & Pitfalls

1. ⚠️ **Parsing logs with `split()`.** A space inside any field shifts every later column
   — and it *succeeds*, producing wrong numbers with no error (section 3.1).
2. ⚠️ **Letting one malformed line kill the run.** `int("‑")` two hours into a batch job
   throws away everything. Parse at the boundary, return `None`, log and count (**15.10**).
3. ⚠️ **`groupby` without sorting** by the grouping key — silent fragmented groups
   (**7.3**). This pipeline gets away with hour-grouping only because logs are
   time-ordered on disk.
4. **Re-compiling the regex per line**, or leaving quantifiers greedy and nested — the
   ReDoS territory of **9.1**. Compile once; keep quantifiers bounded and non-greedy.
5. **Aggregating raw paths.** `/api/jobs/4821` as a key gives one group per request;
   normalise first, then group.
6. **Consuming a generator twice.** A second loop over an exhausted generator yields
   nothing — build a fresh pipeline per pass, as `records()` does here (**4.3**).
7. **Computing percentiles from a mean-friendly summary.** You cannot get p95 from a
   running average; percentiles need the distribution (or a sketch such as t-digest).
8. **Forgetting `newline=""`** when writing CSV — blank rows on Windows (**8.2**).
9. **Module-level mutable stats.** A global `Counter` shared by every pass double-counts
   on the second pass; pass stats in, per pass.

## Best Practices

- Convert types **once, at the parse boundary**; everything downstream gets typed,
  frozen records with units baked into the field name (`latency_ms`).
- Make each stage a small generator with one job; compose with plain iteration,
  `islice`, and generator expressions — testable in isolation, streaming end to end.
- Treat malformed input as **data about your system** — log it with position, count it,
  report it, never crash on it.
- Report **p50/p95**, not means, for anything latency-shaped.
- Emit machine artifacts as **JSON Lines**, human artifacts as **CSV**; write both to a
  temp/output directory, never next to your source.
- Keep the core parse/normalise/aggregate functions in an importable module and cover
  the parser's failure modes in the same suite as its successes (**15.3**).

## Extension exercises

1. Wrap the pipeline in an **argparse CLI**: `logpipe report access.log --format jsonl
   --out reports/` (see folder **17** for packaging it as a console script).
2. Accept **gzip** input transparently — `gzip.open(path, "rt")` when the suffix is
   `.gz`; only `read_lines` should change.
3. Add a **Top-N slowest requests** table (`heapq.nlargest(10, records(),
   key=lambda r: r.latency_ms)` — constant memory, no sort).
4. Harden the regex against hostile input: reread the **ReDoS** section of **9.1**, then
   try to craft a line that makes `LOG_RE` backtrack badly. Why do the bounded
   quantifiers and the `[^\]]`-style negated classes make it hard?
5. Extend `parse_records` to also yield the malformed lines to a **dead-letter file**
   for later inspection — the pattern real ingest systems use.

In [ ]:
# Cleanup: remove every temp file this notebook created.
shutil.rmtree(WORK, ignore_errors=True)
print("removed", WORK)